In [62]:
import numpy as np  #numerical operation and storage in python
import pandas as pd #provides data structures
from scikeras.wrappers import KerasClassifier
from sklearn.feature_extraction.text import TfidfVectorizer #Term Frequency-Inverse Document Frequency Vectorizer
#used to convert a collection of raw text documents into a matrix of TF-IDF features
#This matrix can then be used as input to machine learning models for tasks like text classification, clustering, or information retrieval.
from sklearn.model_selection import train_test_split  # split the dataset into training and testing sets
from sklearn.naive_bayes import MultinomialNB   #this is a Naive Bayes classifier for multinomial models, it is used for text classification
from sklearn.linear_model import LogisticRegression # refers to a statistical method for modeling the probability of a binary outcome
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier, ExtraTreesClassifier, RandomForestClassifier
#an ensemble learning method that belongs to the family of decision tree-based models. It is used for classification tasks, where the goal is to predict the class or category of an input based on its features.
# Like the Random Forest, it's an ensemble of decision trees, but it builds trees sequentially in a way that corrects the errors of the previous ones.
#Gradient Boosting is a form of boosting, a technique where weak learners (typically shallow decision trees) are combined to create a strong learner.
#ensemble learning method in scikit-learn that allows you to combine multiple individual classifiers and aggregate their predictions to make a final prediction
from sklearn.svm import SVC #machine learning algorithm used for both classification and regression tasks
# supervised learning algorithm used for classification tasks. It belongs to the family of instance-based, or memory-based, learning algorithms.
#The fundamental idea behind the K-Nearest Neighbors (KNN) algorithm is to classify a data point based on the majority class of its K nearest neighbors in the feature space.
from nltk.corpus import stopwords   #database( ‘the', 'or', 'and')
from sklearn.metrics import log_loss    #logarithmic loss measures the performance of a classification model where the prediction output is a probability value between 0 and 1
from nltk.stem import PorterStemmer     #process of reducing a word to its base or root form such as reducing -> reduce
from sklearn.model_selection import cross_val_score     #resampling procedure used to evaluate machine learning models on a limited data sample
#The function returns an array of scores, where each score corresponds to the performance of the model on a particular fold.
from sklearn.metrics import precision_score, recall_score, f1_score
#precision_score evaluate the performance of a classification model, particularly for binary or multiclass classification problems. It measures the accuracy of the positive predictions made by the model.
#recall_score measures the ability of the classifier to capture all the positive instances in the dataset
#f1_score the F1 score considers both false positives and false negatives and is particularly useful when there is an uneven class distribution (imbalanced classes)
#It is the harmonic mean of precision and recall and provides a balance between these two metrics.
from sklearn.metrics import hamming_loss,accuracy_score
#the average Hamming distance between the predicted labels and the true labels
#accuracy_score measures the proportion of correctly predicted instances among the total number of instances
from tensorflow.keras.models import Sequential
#The Sequential model is a linear stack of layers, where you can simply add one layer at a time.
#build neural networks
from tensorflow.keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
#dense implement fully connected layers. every node in the layer is connected to every node in the previous layer.
#Embedding transform input data into dense vectors of fixed size.  handling categorical data or sequences, like words in a sentence
#LSTM type of recurrent neural network (RNN) architecture designed to overcome the limitations of traditional RNNs in capturing long-range dependencies in sequences.
#type of dropout layer specifically designed for 1D spatial data, such as sequences in natural language processing or time series data. Dropout is a regularization technique that helps prevent overfitting by randomly setting a fraction of input units to zero during training.
from tensorflow.keras.preprocessing.text import Tokenizer
#Tokenizer convert a collection of text documents into a matrix of token indices.
from tensorflow.keras.preprocessing.sequence import pad_sequences
# The purpose of pad_sequences is to ensure that all sequences in a dataset have the same length by either padding or truncating them.
from tensorflow.keras.callbacks import EarlyStopping
#monitor a specified metric (such as validation loss or accuracy) during training and stop training if the metric stops improving or worsens.
from tensorflow.keras import regularizers
#regularizersprevent overfitting by adding a penalty term to the loss function. Regularization encourages the model to learn simpler patterns and avoid fitting the training data too closely.
import string
import nltk

In [63]:
nltk.download('stopwords')

# Import numpy, pandas, scikit-learn, nltk, and string


stop_words = set(stopwords.words('english')) #download the dataset of stopwords
ps = PorterStemmer()    #define PorterStemmer as ps

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [64]:
def preprocess_text(text):
    if not text:    # if it is none and return " " empty
        return ''
    tokens = [ps.stem(word) for word in text.split() if word not in stop_words and word not in string.punctuation]
    return ' '.join(tokens)
#ps.stem is reducing the words
#text.split is used to split the word into a list of words
#word not in stop_words:  Filters out common stop words using the stop_words set
#word not in string.punctuation: Filters out punctuation marks.
#Joins the processed tokens into a string, separated by spaces. This is done to convert the list of words back into a coherent text string.

In [65]:
# Load the dataset
csv_fake = r"C:\Users\User\OneDrive\文档\AIT(PYTHON AND TENSORFLOW)\Revision Python\True.csv"
csv_true = r"C:\Users\User\OneDrive\文档\AIT(PYTHON AND TENSORFLOW)\Revision Python\Fake.csv"
data_fake = pd.read_csv(csv_fake)   #read the CSV file
data_true = pd.read_csv(csv_true)

In [66]:
print(data_fake.head())
print(data_true.head())

                                               title  \
0  As U.S. budget fight looms, Republicans flip t...   
1  U.S. military to accept transgender recruits o...   
2  Senior U.S. Republican senator: 'Let Mr. Muell...   
3  FBI Russia probe helped by Australian diplomat...   
4  Trump wants Postal Service to charge 'much mor...   

                                                text       subject  \
0  WASHINGTON (Reuters) - The head of a conservat...  politicsNews   
1  WASHINGTON (Reuters) - Transgender people will...  politicsNews   
2  WASHINGTON (Reuters) - The special counsel inv...  politicsNews   
3  WASHINGTON (Reuters) - Trump campaign adviser ...  politicsNews   
4  SEATTLE/WASHINGTON (Reuters) - President Donal...  politicsNews   

                 date  
0  December 31, 2017   
1  December 29, 2017   
2  December 31, 2017   
3  December 30, 2017   
4  December 29, 2017   
                                               title  \
0   Donald Trump Sends Out Embarrassing Ne

In [67]:
data_fake["class"] = 0  #define fake as 0 and true as 1
data_true["class"] = 1

In [68]:
data_fake_manual_testing = data_fake.tail(10)   #get the last ten rows of the news as a manual testing
for i in range(23470, 23470, -1):
    data_fake.drop([i], axis=0, inplace=True)   #delete the last ten rows

data_true_manual_testing = data_true.tail(10)
for i in range(21416, 21406, -1):
    data_true.drop([i], axis=0, inplace=True)

In [69]:
print(data_fake.shape, data_true.shape)     #(23471, 5) (21407, 5)

(21417, 5) (23471, 5)


In [70]:
data_fake_manual_testing["class"] = 0 #false as 0
data_true_manual_testing["class"] = 1  #true as 1

C:\Users\User\AppData\Local\Temp\ipykernel_3976\4039238093.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_fake_manual_testing["class"] = 0 #false as 0
C:\Users\User\AppData\Local\Temp\ipykernel_3976\4039238093.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_true_manual_testing["class"] = 1  #true as 1


In [71]:
data_fake_manual_testing.head(10) #disply he head
data_true_manual_testing.head(10)
# """
# data_manual_testing = pd.concat([data_fake_manual_testing,data_true_manual_testing], axis = 0)
# #Concatenates the "data_fake_manual_testing" and "data_true_manual_testing" DataFrames along the rows (axis=0) to create a single DataFrame for manual testing.
# data_manual_testing.to_csv("manual_testing.csv")
# #Saves the concatenated manual testing DataFrame to a CSV file named "manual_testing.csv."
# """

,title,text,subject,date,class
23471,Seven Iranians freed in the prisoner swap have...,"21st Century Wire says This week, the historic...",Middle-east,"January 20, 2016",1
23472,#Hashtag Hell & The Fake Left,By Dady Chery and Gilbert MercierAll writers ...,Middle-east,"January 19, 2016",1
23473,Astroturfing: Journalist Reveals Brainwashing ...,Vic Bishop Waking TimesOur reality is carefull...,Middle-east,"January 19, 2016",1
23474,The New American Century: An Era of Fraud,Paul Craig RobertsIn the last years of the 20t...,Middle-east,"January 19, 2016",1
23475,Hillary Clinton: ‘Israel First’ (and no peace ...,Robert Fantina CounterpunchAlthough the United...,Middle-east,"January 18, 2016",1
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",1
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",1
23478,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",1
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",1
23480,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016",1


In [72]:
data_merge = pd.concat([data_fake.sample(5000), data_true.sample(5000)], axis =0 )
data_merge.head(10)

,title,text,subject,date,class
9982,"Grassley, Garland reprise 1990s judicial confi...",WASHINGTON (Reuters) - Chuck Grassley and Merr...,politicsNews,"April 11, 2016",0
4930,Trump barnstorms to push healthcare plan; sign...,(Reuters) - U.S. President Donald Trump used h...,politicsNews,"March 14, 2017",0
3490,Kathy Griffin loses CNN deal after photos with...,LOS ANGELES (Reuters) - CNN fired comedian Kat...,politicsNews,"May 31, 2017",0
3716,Ryan: Special counsel will not interfere with ...,WASHINGTON (Reuters) - U.S. House of Represent...,politicsNews,"May 18, 2017",0
6224,Obama suggests U.S. embassy move to Jerusalem ...,WASHINGTON (Reuters) - President Barack Obama ...,politicsNews,"January 18, 2017",0
17276,"Russian, Iranian diplomats to discuss Iran nuc...",MOSCOW (Reuters) - Russian Deputy Foreign Mini...,worldnews,"October 17, 2017",0
2680,N.J. Senator Menendez seeks dismissal of corru...,NEW YORK (Reuters) - Democratic Senator Bob Me...,politicsNews,"July 19, 2017",0
5006,"Trump economic adviser: Fed doing 'good job,' ...",WASHINGTON (Reuters) - White House economic ad...,politicsNews,"March 12, 2017",0
15370,Trump warns 'rogue regime' North Korea of grav...,BEIJING (Reuters) - U.S. President Donald Trum...,worldnews,"November 8, 2017",0
20117,UK terrorism arrests soar to record level afte...,LONDON (Reuters) - The number of people arrest...,worldnews,"September 14, 2017",0


In [73]:
print(data_merge.columns)

Index(['title', 'text', 'subject', 'date', 'class'], dtype='object')


In [74]:
data = data_merge.drop(["title", "subject","date"], axis = 1)
#Drops the specified columns ("title", "subject", "date") from the merged DataFrame
data = data.sample(frac = 1,random_state=42)#Shuffles the rows of the DataFrame randomly using the sample method with frac=1 (indicating to use all rows).

In [75]:
data.head()

,text,class
11638,It seems that with each new retail store that ...,1
11869,SANTIAGO (Reuters) - When Chile s President Mi...,0
15621,"PALERMO, Italy (Reuters) - Silvio Berlusconi s...",0
9558,HAVANA (Reuters) - Cuba and the United States ...,0
13040,MOSCOW (Reuters) - The State Duma lower house ...,0


In [76]:
data.reset_index(inplace = True) #Resets the index of the DataFrame in place.
data.drop(["index"], axis = 1, inplace = True) #Drops the column "index" from the DataFrame in place.

In [77]:
print(data.columns)
print(data.head())

Index(['text', 'class'], dtype='object')
                                                text  class
0  It seems that with each new retail store that ...      1
1  SANTIAGO (Reuters) - When Chile s President Mi...      0
2  PALERMO, Italy (Reuters) - Silvio Berlusconi s...      0
3  HAVANA (Reuters) - Cuba and the United States ...      0
4  MOSCOW (Reuters) - The State Duma lower house ...      0


In [78]:
data["text"] = data["text"].apply(preprocess_text) #apply the preprocess function to clear uselss thing

In [79]:
# Extract features and labels
x = np.array(data["text"])
y = np.array(data["class"])

In [80]:
# Initialize TfidfVectorizer with ngram_range
tfidf = TfidfVectorizer(
    max_features=3000,  # Limit the number of features (vocabulary size) to the top 5000 by term frequency across the corpus.
    min_df=3,           # Ignore terms that have a document frequency strictly lower than 5 (terms that appear in fewer than 5 documents).
    max_df=0.7,         # Ignore terms that have a document frequency strictly higher than 70% (terms that appear in more than 70% of the documents).
    ngram_range=(1, 2)) # Use unigrams and bigrams (sequences of one or two words) as features.
#Defines the range of n-grams to be extracted. In this case, it considers both unigrams (individual words) and bigrams (sequences of two words).

In [81]:
x_tfidf = tfidf.fit_transform(x)
#fits the vectorizer to the given text data (x) and transforms it into a sparse matrix of TF-IDF features (x_tfidf).

In [82]:
# Split the data into training and testing sets
xtrain, xtest, ytrain, ytest = train_test_split(
    x_tfidf, #matrix to be split.
    y,              #corresponding labels.
    test_size=0.3, #30% of the data will be used for testing.
    random_state=42) #ensure reproducibility. Setting a random seed
xtrain_dense = xtrain.todense()
xtest_dense = xtest.todense()
#convert a sparse matrix to a dense matrix.
#In a sparse matrix, most of the elements are zero, and only non-zero elements are stored along with their indices.
#On the other hand, a dense matrix stores all elements, regardless of whether they are zero or not.

In [83]:
# Convert the dense matrix to a list of texts
xtrain_texts = [' '.join(map(str, row)) for row in xtrain_dense]    #map(str, row) converts each element to a string
xtest_texts = [' '.join(map(str, row)) for row in xtest_dense]      #join(...) concatenates these strings with space as the separator

In [84]:
tokenizer = Tokenizer(num_words=5000, split=" ")
#This parameter specifies the maximum number of words to keep, based on word frequency. In this case, it limits the vocabulary to the top 500 most frequent words.
#split=" ": This parameter specifies the word separator. In this case, words are split based on space.
tokenizer.fit_on_texts(xtrain_texts)
#it processes the training texts (xtrain_texts) and builds a word index.
#The word index is a dictionary where words are keys and their corresponding integer indices are values.

In [85]:
xtrain_seq = tokenizer.texts_to_sequences(xtrain_texts)
#converts a list of texts to a list of sequences, where each word in the text is replaced by its corresponding integer index in the word index created during fitting.
xtest_seq = tokenizer.texts_to_sequences(xtest_texts)
#converts a list of texts to a list of sequences, where each word in the text is replaced by its corresponding integer index in the word index created during fitting.


In [86]:
max_len = max(len(x) for x in xtrain_seq)
#This line calculates the maximum length of sequences in the training set.
#This information is used later to pad or truncate sequences to a uniform length.
xtrain_pad = pad_sequences(xtrain_seq, maxlen=max_len)
xtest_pad = pad_sequences(xtest_seq, maxlen=max_len)
# pad_sequences ensure that all sequences in the training and testing sets have the same length.
# It pads shorter sequences with zeros or truncates longer sequences to achieve the specified maxlen.

In [87]:
# Build the neural network model
def create_lstm_model():
    seq = Sequential() #build a model layer by layer in a step-by-step fashion.
    seq.add(Embedding(input_dim=5000, output_dim=128, input_length=max_len)) #used for word embeddings. It turns positive integers (indexes) into dense vectors of fixed size.
    seq.add(SpatialDropout1D(0.2))  #Spatial 1D version of dropout. It drops entire 1D feature maps, with a dropout rate of 0.2.
    #Long Short-Term Memory layer, a type of recurrent layer in neural networks. It helps the model capture long-term dependencies in the input sequence
    seq.add(LSTM(100,#Defines a Long Short-Term Memory layer with 100 memory units.
                 dropout=0.2,   #Applies dropout to the input units with a probability of 0.2 during training, helping to prevent overfitting.
                 recurrent_dropout=0.2, #Applies dropout to the recurrent units with a probability of 0.2 during training, also aiding in preventing overfitting.
                 return_sequences=True,  #This is set to True when you have subsequent LSTM layers. It returns the full sequence of outputs for each timestep.
                 kernel_regularizer=regularizers.l2(0.01))) #Applies L2 regularization to the weights matrix.

    seq.add(LSTM(50,#50 memory units
                 dropout=0.2,
                 recurrent_dropout=0.2,
                 kernel_regularizer=regularizers.l2(0.01)))

    seq.add(Dense(1,#Adds a dense layer with 1 neuron, suitable for binary classification.
                  activation='sigmoid'))#Applies the sigmoid activation function, squashing the output between 0 and 1.
    # Compile the model
    seq.compile(loss='binary_crossentropy', #Binary cross-entropy is chosen as the loss function, suitable for binary classification problems.
                optimizer='adam',#Adam optimizer is used for training.
                metrics=['accuracy'])#The model will track accuracy during training.
    return seq

In [88]:
# Create a KerasClassifier instance with EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', #Monitors the validation loss during training.
                               patience=3,  #If the validation loss does not improve for 3 consecutive epochs, training will be stopped.
                               restore_best_weights=True)# Restores the weights from the epoch with the best value of the monitored quantity (in this case, validation loss).


In [89]:
lstm_classifier = KerasClassifier(build_fn=create_lstm_model, #Specifies the function to build the Keras model (previously defined create_lstm_model function).
                                  epochs=5, #Number of epochs for training.
                                  batch_size=32,    #Number of samples per gradient update during training.
                                  verbose=0,    #Training progress is not displayed.
                                  callbacks=[early_stopping])#Incorporates the EarlyStopping callback during training.


In [90]:
# Fit the model
lstm_classifier.fit(xtrain_pad, ytrain,validation_data=(xtest_pad, ytest))
#The validation data to monitor during training. It consists of:
#   xtest_pad: The padded sequences of the test data.
#   ytest: The corresponding labels for the test data.

c:\Users\User\anaconda3\envs\ML\lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


KerasClassifier(
	model=None
	build_fn=<function create_lstm_model at 0x000001B4793EEB90>
	warm_start=False
	random_state=None
	optimizer=rmsprop
	loss=None
	metrics=None
	batch_size=32
	validation_batch_size=None
	verbose=0
	callbacks=[<keras.src.callbacks.EarlyStopping object at 0x000001B405309A50>]
	validation_split=0.0
	shuffle=True
	run_eagerly=False
	epochs=5
	class_weight=None
)

In [91]:
# Evaluate the model
y_pred_prob = lstm_classifier.predict_proba(xtest_pad).astype(float)  # Use predict_proba instead of predict
#predict_proba  predict class probabilities for the input test data (xtest_pad).
#The .astype(float) is used to ensure the values are of the float data type.


In [92]:
y_pred = np.round(y_pred_prob[:, 1]).astype(int)#The predicted probabilities are rounded to obtain the predicted class labels
#it rounds the probability of the positive class (index 1) to either 0 or 1. The resulting array is then converted to integers.


In [93]:
# Convert string labels to numeric values
ytest_numeric = np.array(ytest)
#Converts the true labels of the test set (ytest) to a NumPy array. This step might be unnecessary if ytest is already a NumPy array.

y_pred_numeric = np.round(y_pred).flatten()
#Ensures that the predicted labels are in the same format as the true labels by rounding and flattening the array.
# This is done to facilitate the comparison of predicted and true labels in the subsequent evaluation metrics.


In [94]:
# Print evaluation metrics for the LSTM model
precision_seq = precision_score(ytest_numeric, y_pred, average='weighted')
recall_seq = recall_score(ytest_numeric, y_pred, average='weighted')
f1_seq = f1_score(ytest_numeric, y_pred, average='weighted')
hamming_seq = hamming_loss(ytest_numeric, y_pred)
accuracy_seq = accuracy_score(ytest_numeric, y_pred)
"""score_seq = cross_val_score(lstm_classifier, xtest, ytest, cv=5)"""
log_loss_seq = log_loss(ytest_numeric, y_pred_prob)

c:\Users\User\anaconda3\envs\ML\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\User\anaconda3\envs\ML\lib\site-packages\sklearn\metrics\_classification.py:2922: UserWarning: The y_pred values do not sum to one. Starting from 1.5 thiswill result in an error.
  warnings.warn(


In [95]:
print(f"Sequential Log Loss: {log_loss_seq}")
print(f"Sequential Precision: {precision_seq}")
print(f"Sequential Recall: {recall_seq}")
print(f"Sequential F1-Score: {f1_seq}")
print(f"Sequential Haming Loss: {hamming_seq}")
"""print(f"Sequential Score: {score_seq}")"""
print(f"Sequential Accuracy: {accuracy_seq}")

Sequential Log Loss: 0.6931240115253001
Sequential Precision: 0.26316900000000004
Sequential Recall: 0.513
Sequential F1-Score: 0.34787706543291474
Sequential Haming Loss: 0.487
Sequential Accuracy: 0.513


In [96]:
# Initialize classifiers
classifiers = {
    'Multinomial Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(),
    'Support Vector Machine': SVC(),
    'Extra Trees': ExtraTreesClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'Random Forest': RandomForestClassifier()
}


In [97]:
# Train and evaluate each classifier
for name, model in classifiers.items():
    """if name == 'Random Forest':
        # Grid search for RandomForestClassifier
        rf_param_grid = {
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }

        grid_search = GridSearchCV(model, rf_param_grid, cv=5, scoring='accuracy')
        grid_search.fit(xtrain, ytrain)

        print(f"{name} Best Parameters:", grid_search.best_params_)
        print(f"{name} Best Accuracy:", grid_search.best_score_)

        # Use the best RandomForestClassifier
        classifiers[name] = grid_search.best_estimator_
    else:"""
    train = model.fit(xtrain, ytrain)   #fits the model using training data
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(xtest)
        loss = log_loss(ytest, y_prob)
        print(f"{name} Log Loss: {loss}")
        y_pred = model.predict(xtest)  # Predictions for classification
        # uses average='weighted' to compute precision
        precision = precision_score(ytest, y_pred,
                                        average='weighted')  # Precision measures the accuracy of the positive predictions
        recall = recall_score(ytest, y_pred,
                                  average='weighted')  # recall measures the ability of the classifier to capture all the positive instances
        f1 = f1_score(ytest, y_pred, average='weighted')  # F1-score is the harmonic mean of precision and recall.
        hamming = hamming_loss(ytest, y_pred)  # Measures the fraction of labels that are incorrectly predicted.

        print(f"{name} Precision: {precision}")
        print(f"{name} Recall: {recall}")
        print(f"{name} F1-Score: {f1}")
        print(f"{name} Haming: {hamming}")

        accuracy = model.score(xtest, ytest)  # Computes and prints the accuracy of the model on the test set
        scores = cross_val_score(model, xtest, ytest, cv=5)
        print(f"{name} Score: {scores}")
        print(f"{name} Accuracy: {accuracy}")
        print("--------------------------------------------")
    else:
        print(f"{name} does not support log loss computation.")
        y_pred = model.predict(xtest)  # Predictions for classification
        # uses average='weighted' to compute precision
        precision = precision_score(ytest, y_pred,
                                    average='weighted')  # Precision measures the accuracy of the positive predictions
        recall = recall_score(ytest, y_pred,
                              average='weighted')  # recall measures the ability of the classifier to capture all the positive instances
        f1 = f1_score(ytest, y_pred, average='weighted')  # F1-score is the harmonic mean of precision and recall.
        hamming = hamming_loss(ytest, y_pred)  # Measures the fraction of labels that are incorrectly predicted.
        print(f"{name} Precision: {precision}")
        print(f"{name} Recall: {recall}")
        print(f"{name} F1-Score: {f1}")
        print(f"{name} Haming: {hamming}")

        accuracy = model.score(xtest, ytest)  # Computes and prints the accuracy of the model on the test set
        scores = cross_val_score(model, xtest, ytest, cv=5)
        print(f"{name} Score: {scores}")
        print(f"{name} Accuracy: {accuracy}")
        print("--------------------------------------------")

c:\Users\User\anaconda3\envs\ML\lib\site-packages\sklearn\metrics\_classification.py:2922: UserWarning: The y_pred values do not sum to one. Starting from 1.5 thiswill result in an error.
  warnings.warn(


Multinomial Naive Bayes Log Loss: 0.1563837147899277
Multinomial Naive Bayes Precision: 0.9493333333333334
Multinomial Naive Bayes Recall: 0.9493333333333334
Multinomial Naive Bayes F1-Score: 0.9493333333333334
Multinomial Naive Bayes Haming: 0.050666666666666665
Multinomial Naive Bayes Score: [0.925      0.91666667 0.93833333 0.95       0.93666667]
Multinomial Naive Bayes Accuracy: 0.9493333333333334
--------------------------------------------
Logistic Regression Log Loss: 0.1284021372452286
Logistic Regression Precision: 0.9850066923272579
Logistic Regression Recall: 0.985
Logistic Regression F1-Score: 0.9850006086937477
Logistic Regression Haming: 0.015
Logistic Regression Score: [0.965      0.965      0.98166667 0.985      0.97333333]
Logistic Regression Accuracy: 0.985
--------------------------------------------
Support Vector Machine does not support log loss computation.
Support Vector Machine Precision: 0.9910003744625441
Support Vector Machine Recall: 0.991
Support Vector Ma

In [98]:
# Ensemble method: Voting Classifier
vote = VotingClassifier(estimators=list(classifiers.items()), voting='hard')
#ensemble learning method in scikit-learn that allows you to combine multiple individual classifiers and aggregate their predictions.
#estimators=list(classifiers.items()): This parameter specifies the list of base classifiers
#voting='hard': This parameter specifies that the ensemble will use "hard" voting,
#where the predicted class is determined by a majority vote among the base classifiers.
train_vote = vote.fit(xtrain, ytrain)
#train the ensemble model using the training data


In [99]:
y_pred = vote.predict(xtest)  # Predictions for classification
# uses average='weighted' to compute precision
precision_vote = precision_score(ytest, y_pred,
                            average='weighted')  # Precision measures the accuracy of the positive predictions
recall_vote = recall_score(ytest, y_pred,
                      average='weighted')  # recall measures the ability of the classifier to capture all the positive instances
f1_vote = f1_score(ytest, y_pred, average='weighted')  # F1-score is the harmonic mean of precision and recall.
hamming_vote = hamming_loss(ytest, y_pred)  # Measures the fraction of labels that are incorrectly predicted.

In [100]:
print(f"Voting Classifier Precision: {precision_vote}")
print(f"Voting Classifier Recall: {recall_vote}")
print(f"Voting Classifier F1-Score: {f1_vote}")
print(f"Voting Classifier Hamming: {hamming_vote}")

Voting Classifier Precision: 0.994350262668973
Voting Classifier Recall: 0.9943333333333333
Voting Classifier F1-Score: 0.9943328399183714
Voting Classifier Haming: 0.005666666666666667


In [101]:
accuracy_vote = vote.score(xtest, ytest)  # Computes and prints the accuracy of the model on the test set
scores_vote = cross_val_score(vote, xtest, ytest, cv=5)
print(f"Voting Classifier Score: {scores_vote}")
print(f"Voting Classifier Accuracy: {accuracy_vote}")
print("--------------------------------------------")

Voting Classifier Score: [0.985      0.985      0.99       0.99333333 0.98833333]
Voting Classifier Accuracy: 0.9943333333333333
--------------------------------------------


In [102]:
# User input for predicting new headlines
news_headline = str(input("Enter the news name: "))
# Preprocess the input headline
processed_headline = preprocess_text(news_headline)
# Transform the processed headline into a TF-IDF vector
data = tfidf.transform([processed_headline])

In [103]:
# Make predictions using each classifier
for name, model in classifiers.items():
    prediction = model.predict(data)
    if prediction == 0:
        prediction = "FALSE"
    else:
        prediction = "True"
    print(f"{name} Prediction: {prediction}")


Multinomial Naive Bayes Prediction: True
Logistic Regression Prediction: True
Support Vector Machine Prediction: True
Extra Trees Prediction: True
Gradient Boosting Prediction: True
Random Forest Prediction: True


In [104]:
# Make a prediction using the Voting Classifier
voting_prediction = vote.predict(data)  #predict the data
if voting_prediction == 0:
    voting_prediction = "FALSE"
else:
    voting_prediction = "TRUE"
print(f"Voting Classifier Prediction: {voting_prediction}")


Voting Classifier Prediction: TRUE
